# Domain 1: Agentic Architecture & Orchestration
## TS 1.2: Orchestrate Multi-Agent Systems with Coordinator-Subagent Patterns
**Certification Focus**: CCAR-F Domain 1 (27% Exam Weight) | **Scenario Alignment**: Scenario 3 (Multi-Agent Research System)

---

### 🎯 Key Architectural Principles
1. **Hub-and-Spoke Topology**: All inter-subagent communication, auditing, routing, and synthesis flow strictly through the coordinator. Subagents never communicate peer-to-peer directly.
2. **Context Isolation Contract**: Subagents **do not** inherit the coordinator's message history, memory, or global permissions. Each subagent invocation starts with an isolated, pristine context.
3. **Query-Complexity Routing**: Simple queries are answered directly by the coordinator; complex queries are routed to specialized subagents.
4. **Iterative Refinement & Gap Detection**: The coordinator audits subagent outputs against query goals and re-delegates follow-up tasks if coverage is incomplete.

In [ ]:
# Step 1: Environment & Offline Setup
import json
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from lib.mock_claude import MockClaudeClient, MockContentBlock, MockMessageResponse

print(" Environment initialized for TS 1.2 Coordinator-Subagent Pattern.")

### 👥 Step 2: Specialized Subagent Definitions (Isolated Contexts)
Each subagent has a dedicated specialization and receives only the explicit parameters required for its job.

In [ ]:
# Step 2: Isolated Subagent Classes
class WebSearchSubagent:
    """Subagent specialized in retrieving external research snippets."""
    KB = {
        "claude agent sdk": "Claude Agent SDK provides primitives for building agentic loops, subagent delegation, hooks, and session resumption.",
        "mcp error handling": "MCP errors should return isError: true with categorized codes (transient, validation, business).",
    }

    def execute(self, query: str) -> Dict[str, Any]:
        q = query.lower()
        matched = []
        for key, val in self.KB.items():
            if any(w in q for w in key.split()):
                matched.append({"topic": key, "snippet": val})
        return {
            "source": "web_search",
            "query": query,
            "found": len(matched) > 0,
            "findings": matched if matched else "No matching external articles.",
        }


class DocumentAnalysisSubagent:
    """Subagent specialized in deep document extraction."""
    def execute(self, doc_id: str, focus_topic: str) -> Dict[str, Any]:
        return {
            "source": "doc_analysis",
            "doc_id": doc_id,
            "focus": focus_topic,
            "metrics": [
                {"metric": "Passing score", "value": "720/1000"},
                {"metric": "Question count", "value": "60 items"},
            ],
        }


class SynthesisSubagent:
    """Subagent specialized in synthesizing multi-source findings."""
    def execute(self, findings: List[Dict[str, Any]], required_topics: List[str]) -> Dict[str, Any]:
        text_dump = json.dumps(findings).lower()
        missing = [t for t in required_topics if t.lower() not in text_dump]
        return {
            "coverage_sufficient": len(missing) == 0,
            "missing_topics": missing,
            "synthesized_findings_count": len(findings),
        }

print(" Subagents declared with isolated execution scopes.")

### 🎛️ Step 3: Multi-Agent Coordinator Implementation
The coordinator orchestrates subagent calls, manages the audit trail, and checks synthesis completeness.

In [ ]:
# Step 3: Hub-and-Spoke Coordinator
class MultiAgentCoordinator:
    def __init__(self, client: Any) -> None:
        self.client = client
        self.web_agent = WebSearchSubagent()
        self.doc_agent = DocumentAnalysisSubagent()
        self.synth_agent = SynthesisSubagent()
        self.audit_log: List[Dict[str, Any]] = []

    def dispatch_tool(self, tool_name: str, inputs: Dict[str, Any]) -> Dict[str, Any]:
        self.audit_log.append({"tool": tool_name, "inputs": inputs})
        if tool_name == "delegate_web_search":
            return self.web_agent.execute(inputs["query"])
        elif tool_name == "delegate_doc_analysis":
            return self.doc_agent.execute(inputs["doc_id"], inputs["focus_topic"])
        elif tool_name == "evaluate_synthesis":
            return self.synth_agent.execute(inputs["findings"], inputs["required_topics"])
        raise ValueError(f"Unknown coordinator tool: {tool_name}")

    def run_research_pipeline(self, topic: str, required_topics: List[str]) -> Dict[str, Any]:
        # Turn 1: Delegate search
        search_res = self.dispatch_tool("delegate_web_search", {"query": topic})
        
        # Turn 2: Delegate doc analysis
        doc_res = self.dispatch_tool("delegate_doc_analysis", {"doc_id": "DOC-2026", "focus_topic": topic})
        
        # Turn 3: Evaluate synthesis coverage
        synth_res = self.dispatch_tool("evaluate_synthesis", {
            "findings": [search_res, doc_res],
            "required_topics": required_topics
        })

        return {
            "status": "completed" if synth_res["coverage_sufficient"] else "gaps_identified",
            "search_res": search_res,
            "doc_res": doc_res,
            "synthesis": synth_res,
            "audit_log": self.audit_log
        }

print(" MultiAgentCoordinator ready for pipeline execution.")

### 🧪 Step 4: Offline Simulation Walkthrough
We execute the coordinator pipeline on a multi-domain research query and observe context isolation and audit logging.

In [ ]:
# Step 4: Run Coordinator Simulation
coord = MultiAgentCoordinator(client=None)

# Run research pipeline
results = coord.run_research_pipeline(
    topic="claude agent sdk",
    required_topics=["claude agent sdk"]
)

print("Pipeline Status:", results["status"])
print("Coverage Sufficient:", results["synthesis"]["coverage_sufficient"])
print("Audit Trail Steps:", len(results["audit_log"]))

### 🔬 Step 5: Verification Assertions
We assert that:
1. Subagents do not cross-contaminate state.
2. Missing topics are accurately flagged during synthesis.
3. Every delegated action is captured in the coordinator's audit log.

In [ ]:
# Step 5: Verification Checks
# 1. Verify complete synthesis coverage
assert results["status"] == "completed"
assert results["synthesis"]["coverage_sufficient"] is True

# 2. Test gap detection with a missing topic
gap_check = coord.synth_agent.execute([results["search_res"]], required_topics=["missing_special_topic"])
assert gap_check["coverage_sufficient"] is False
assert "missing_special_topic" in gap_check["missing_topics"]

# 3. Verify audit log captures all 3 turns
assert len(results["audit_log"]) == 3

print(" All verification assertions passed! Subagent isolation confirmed.")

### 💡 CCAR-F Exam Golden Rules
- **Rule 1**: Subagents must **never** share state, conversation history, or memory. They receive explicit task prompts.
- **Rule 2**: All orchestration and routing happens in the coordinator (hub-and-spoke), never peer-to-peer.
- **Rule 3**: The coordinator must evaluate intermediate outputs against goals and re-delegate when gaps exist.